Cell 1

In [ ]:
# =========================================
# CELL 1 - INSTALL PACKAGES
# Chạy 1 lần khi mới mở Colab
# =========================================
!pip uninstall -y onnxruntime onnxruntime-gpu
!pip install -q ultralytics open_clip_torch==2.24.0 rtmlib onnxruntime-gpu opencv-python-headless scipy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 49.4 MB/s eta 0:00:00


Cell 2

In [ ]:
# =========================================
# CELL 2 - MOUNT DRIVE
# Chạy 1 lần
# =========================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Cell 3

In [ ]:
# =========================================
# CELL 3 - IMPORTS + CONFIG
# Chạy 1 lần
# =========================================
import os
import json
import math
import time
import shutil
import traceback
import subprocess
from collections import deque
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy.optimize import linear_sum_assignment

import torch
import torch.nn as nn
import torch.nn.functional as F
import open_clip
from ultralytics import YOLO
from rtmlib import RTMPose

# ====== MODEL ARTIFACT DIR ======
RUN_DIR = "/content/drive/MyDrive/OVERALL_DATASET_REPORT/VERSION_LATE_Fine/runs/fusion_v3_20260308_005702"

# ====== JOB BRIDGE DIR ======
JOBS_ROOT = "/content/drive/MyDrive/classroom_jobs"
INCOMING_DIR = os.path.join(JOBS_ROOT, "incoming")
PROCESSING_DIR = os.path.join(JOBS_ROOT, "processing")
DONE_DIR = os.path.join(JOBS_ROOT, "done")
FAILED_DIR = os.path.join(JOBS_ROOT, "failed")

for d in [INCOMING_DIR, PROCESSING_DIR, DONE_DIR, FAILED_DIR]:
    os.makedirs(d, exist_ok=True)

# ====== DETECTION ======
DET_WEIGHTS = "yolov8n.pt"
IMG_SIZE = 1280
CONF_TH = 0.10
IOU_TH = 0.5
PAD = 0

# ====== TRACKING ======
NUM_TARGETS = 5
T_BUFFER = 24
N_FRAMES_IMG = 5
N_POSE = 8
PRED_STRIDE = 12
FRAME_STRIDE = 1
MAX_SECONDS = None   # None = full video

BBOX_EMA_BETA = 0.85
GHOST_MAX_GAP = 45
MAX_CENTER_DIST = 80
MIN_IOU_MATCH = 0.05

# ====== LABEL SMOOTHING ======
EMA_BETA = 0.88
SWITCH_K = 3
TH_SWITCH = 0.62
MARGIN_TH = 0.10
TH_UNKNOWN = 0.40

# ====== DISPLAY ======
SHOW_FPS_ON_VIDEO = True
FPS_EMA = 0.9

# ====== LABEL GROUPS ======
NEGATIVE = {"sleeping", "using_phone"}
WARNING = {"turning"}
NORMAL = {"writing", "reading", "raising_hand"}

# ====== PHONE EVENT ======
PHONE_LABEL = "using_phone"
PHONE_CONF_TH = 0.60
SEGMENT_GAP_SEC = 1.0
MIN_SEGMENT_SEC = 0.5
CLIP_DURATION_SEC = 5.0
CLIP_USE_START_AS_ANCHOR = True

# ====== WORKER ======
POLL_INTERVAL_SEC = 5
KEEP_RUNNING = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("RUN_DIR:", RUN_DIR)
print("JOBS_ROOT:", JOBS_ROOT)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Device: cuda
RUN_DIR: /content/drive/MyDrive/OVERALL_DATASET_REPORT/VERSION_LATE_Fine/runs/fusion_v3_20260308_005702
JOBS_ROOT: /content/drive/MyDrive/classroom_jobs


Cell 4

In [ ]:
# =========================================
# CELL 4 - LOAD ARTIFACTS + LOAD MODELS
# Chạy 1 lần
# =========================================

def now_iso():
    return datetime.utcnow().isoformat(timespec="seconds") + "Z"

def write_json(path, data):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

with open(os.path.join(RUN_DIR, "labels.json"), "r", encoding="utf-8") as f:
    LABELS = json.load(f)

with open(os.path.join(RUN_DIR, "prompt_groups.json"), "r", encoding="utf-8") as f:
    PROMPT_GROUPS = json.load(f)

text_features_prompts = torch.load(
    os.path.join(RUN_DIR, "text_features_prompts.pt"),
    map_location="cpu"
).to(device)

class PoseEncoderJointToken(nn.Module):
    def __init__(self, T=24, J=17, d=256, depth=4, nhead=8, dropout=0.1):
        super().__init__()
        self.T, self.J, self.d = T, J, d
        self.in_proj = nn.Linear(3, d)
        self.joint_emb = nn.Embedding(J, d)
        self.time_emb = nn.Embedding(T, d)
        layer = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=nhead,
            dim_feedforward=4 * d,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.tr = nn.TransformerEncoder(layer, num_layers=depth)

    def forward(self, kpts):
        B, T, J, C = kpts.shape
        x = self.in_proj(kpts)
        jt = torch.arange(J, device=kpts.device)
        tt = torch.arange(T, device=kpts.device)
        x = x + self.joint_emb(jt)[None, None, :, :] + self.time_emb(tt)[None, :, None, :]
        x = x.reshape(B, T * J, self.d)
        x = self.tr(x)
        return x.mean(dim=1)

class AlphaGatedFusionV2(nn.Module):
    def __init__(self, d=512, r_pose_dim=5, r_img_dim=2, hidden=512, dropout=0.1):
        super().__init__()
        self.r_norm = nn.LayerNorm(r_pose_dim + r_img_dim)
        in_dim = 3 * d + (r_pose_dim + r_img_dim)
        self.gate = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )

    def forward(self, f_pose, f_img, r_pose, r_img):
        diff = torch.abs(f_pose - f_img)
        r = torch.cat([r_pose, r_img], dim=-1)
        r = self.r_norm(r)
        x = torch.cat([f_pose, f_img, diff, r], dim=-1)
        alpha = torch.sigmoid(self.gate(x))
        f = alpha * f_pose + (1 - alpha) * f_img
        return f, alpha

class FusionCLIPClassifier(nn.Module):
    def __init__(self, text_feats_prompts, prompt_groups, T=24, J=17, pose_d=256, clip_d=512):
        super().__init__()
        self.pose_enc = PoseEncoderJointToken(T=T, J=J, d=pose_d)
        self.pose_to_clip = nn.Sequential(nn.LayerNorm(pose_d), nn.Linear(pose_d, clip_d))
        self.fusion = AlphaGatedFusionV2(d=clip_d, r_pose_dim=5, r_img_dim=2)
        self.register_buffer("text_feats", text_feats_prompts)
        self.prompt_groups = prompt_groups
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)), requires_grad=False)

    def pool_classes(self, sim_prompt):
        outs = []
        for idxs in self.prompt_groups:
            idxs = list(map(int, idxs))
            s = sim_prompt[:, idxs]
            outs.append(torch.logsumexp(s, dim=1) - math.log(len(idxs)))
        return torch.stack(outs, dim=1)

    def forward(self, kpts, img_emb, r_pose, r_img):
        f_pose = self.pose_enc(kpts)
        f_pose = self.pose_to_clip(f_pose)
        f_pose = F.normalize(f_pose, dim=-1)
        f_img = F.normalize(img_emb, dim=-1)
        f_fuse, alpha = self.fusion(f_pose, f_img, r_pose, r_img)
        f_fuse = F.normalize(f_fuse, dim=-1)
        scale = self.logit_scale.exp().clamp(1.0, 100.0)
        sim_prompt = scale * (f_fuse @ self.text_feats.t())
        logits = self.pool_classes(sim_prompt)
        return logits, alpha

print("Loading fusion model...")
model = FusionCLIPClassifier(
    text_features_prompts,
    PROMPT_GROUPS,
    T=T_BUFFER,
    J=17
).to(device).eval()

ckpt = torch.load(os.path.join(RUN_DIR, "best_model.pth"), map_location=device)
model.load_state_dict(ckpt, strict=True)

print("Loading CLIP...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)
clip_model = clip_model.to(device).eval()

print("Loading RTMPose...")
RTMPOSE_ONNX = "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.zip"
pose_model = RTMPose(
    onnx_model=RTMPOSE_ONNX,
    model_input_size=(192, 256),
    backend="onnxruntime",
    device=device,
    to_openpose=False
)

print("Loading YOLO...")
yolo = YOLO(DET_WEIGHTS)

print("All models loaded successfully.")

Loading fusion model...


/tmp/ipykernel_666/2943754828.py:45: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tr = nn.TransformerEncoder(layer, num_layers=depth)


Loading CLIP...


100%|████████████████████████████████████████| 354M/354M [00:01<00:00, 195MiB/s]


Loading RTMPose...


Downloading: "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.zip" to /root/.cache/rtmlib/hub/checkpoints/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.zip
100%|██████████| 48.4M/48.4M [00:02<00:00, 20.5MB/s]
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


load /root/.cache/rtmlib/hub/checkpoints/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.onnx with onnxruntime backend
Loading YOLO...
All models loaded successfully.


Cell 5

In [ ]:
# =========================================
# CELL 5 - UTILS + TRACKING + RECOGNITION HELPERS
# Chạy 1 lần
# =========================================

def write_status(job_dir, job_id, session_id, status, progress=0, message="", extra=None):
    payload = {
        "job_id": job_id,
        "session_id": session_id,
        "status": status,
        "progress": int(progress),
        "message": message,
        "updated_at": now_iso(),
    }
    if extra:
        payload.update(extra)
    write_json(os.path.join(job_dir, "status.json"), payload)


def blur_score(rgb_img):
    g = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2GRAY)
    return float(cv2.Laplacian(g, cv2.CV_64F).var())


def normalize_kpts_clip(kpts, conf_thr=0.2):
    xy = kpts[..., :2]
    conf = kpts[..., 2:3]
    mask = (conf > conf_thr).float()
    if mask.sum() < 5:
        return kpts
    denom = mask.sum(dim=(0, 1), keepdim=True).clamp(min=1.0)
    mean = (xy * mask).sum(dim=(0, 1), keepdim=True) / denom
    xy = xy - mean
    dist = torch.sqrt(((xy * mask) ** 2).sum(dim=-1, keepdim=True))
    scale = dist.max().clamp(min=1e-3)
    xy = xy / scale
    return torch.cat([xy, conf], dim=-1)


def compute_pose_extra_feats(kpts_norm):
    xy = kpts_norm[..., :2]
    lw = xy[:, 9]
    rw = xy[:, 10]
    lw_spd = (lw[1:] - lw[:-1]).norm(dim=-1).mean()
    rw_spd = (rw[1:] - rw[:-1]).norm(dim=-1).mean()
    wrist_speed = 0.5 * (lw_spd + rw_spd)

    ls = xy[:, 5]
    rs = xy[:, 6]
    vec = rs - ls
    angle = torch.atan2(vec[:, 1], vec[:, 0])
    turn_energy = (angle[1:] - angle[:-1]).abs().mean()
    return wrist_speed, turn_energy


def fill_kpts_to_T(kpts_sparse, idxs, T=24):
    out = np.zeros((T, 17, 3), dtype=np.float32)
    for k, t in enumerate(idxs):
        out[t] = kpts_sparse[k]
    last = None
    for t in range(T):
        if out[t].sum() > 0:
            last = out[t].copy()
        elif last is not None:
            out[t] = last
    if out[0].sum() == 0:
        for t in range(T):
            if out[t].sum() > 0:
                out[:t] = out[t]
                break
    return out


def label_color(label):
    if label in NEGATIVE:
        return (0, 0, 255)
    if label in WARNING:
        return (0, 255, 255)
    if label in NORMAL:
        return (0, 255, 0)
    return (160, 160, 160)


def draw_tag(frame, x1, y1, text, color):
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.5
    thickness = 1
    (tw, th), _ = cv2.getTextSize(text, font, scale, thickness)
    y1 = max(0, y1)
    cv2.rectangle(frame, (x1, y1 - th - 6), (x1 + tw + 6, y1), color, -1)
    cv2.putText(frame, text, (x1 + 3, y1 - 4), font, scale, (0, 0, 0), thickness, cv2.LINE_AA)


def bbox_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = max(1, boxA[2] - boxA[0]) * max(1, boxA[3] - boxA[1])
    areaB = max(1, boxB[2] - boxB[0]) * max(1, boxB[3] - boxB[1])
    return inter / float(areaA + areaB - inter + 1e-6)


def box_center(box):
    return np.array([(box[0] + box[2]) / 2.0, (box[1] + box[3]) / 2.0], dtype=np.float32)


def center_dist(boxA, boxB):
    return float(np.linalg.norm(box_center(boxA) - box_center(boxB)))


def area_of(box):
    return max(1.0, float(box[2] - box[0])) * max(1.0, float(box[3] - box[1]))


def size_penalty(boxA, boxB):
    a = area_of(boxA)
    b = area_of(boxB)
    r = max(a, b) / max(1.0, min(a, b))
    return float(np.log(r + 1e-6))


def cosine_np(a, b):
    na = np.linalg.norm(a) + 1e-8
    nb = np.linalg.norm(b) + 1e-8
    return float(np.dot(a, b) / (na * nb))


def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def ffmpeg_cut_clip(input_video, out_path, start_sec, duration_sec):
    start_sec = max(0.0, float(start_sec))
    duration_sec = max(0.1, float(duration_sec))
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    cmd = [
        "ffmpeg", "-y",
        "-ss", f"{start_sec:.3f}",
        "-i", input_video,
        "-t", f"{duration_sec:.3f}",
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "23",
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
        "-movflags", "+faststart",
        out_path,
    ]
    try:
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return True, ""
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode("utf-8", errors="ignore") if e.stderr else str(e)
        return False, err


def ffmpeg_make_web_video(input_video, output_video):
    os.makedirs(os.path.dirname(output_video), exist_ok=True)

    cmd = [
        "ffmpeg",
        "-y",
        "-i", input_video,
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "23",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-an",
        output_video,
    ]
    try:
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return True, ""
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode("utf-8", errors="ignore") if e.stderr else str(e)
        return False, err


class BBoxFilter:
    def __init__(self, beta_pos=0.88, beta_size=0.96):
        self.beta_pos = beta_pos
        self.beta_size = beta_size
        self.ema_c = None
        self.ema_s = None

    def update(self, box):
        x1, y1, x2, y2 = map(float, box)
        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0
        w = max(2.0, x2 - x1)
        h = max(2.0, y2 - y1)

        c = np.array([cx, cy], dtype=np.float32)
        s = np.array([w, h], dtype=np.float32)

        if self.ema_c is None:
            self.ema_c = c
            self.ema_s = s
        else:
            self.ema_c = self.beta_pos * self.ema_c + (1 - self.beta_pos) * c
            self.ema_s = self.beta_size * self.ema_s + (1 - self.beta_size) * s

        cx, cy = self.ema_c
        w, h = self.ema_s
        return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dtype=np.float32)


class TrackState:
    def __init__(self):
        self.logits_ema = None
        self.label_id = None
        self.label_name = "analyzing..."
        self.conf = 0.0
        self.alpha = 0.0
        self.streak = 0
        self.kpts_last = None

    def update_logits(self, logits_np, alpha, kpts_last):
        self.kpts_last = kpts_last
        self.alpha = float(alpha)

        if self.logits_ema is None:
            self.logits_ema = logits_np
        else:
            self.logits_ema = EMA_BETA * self.logits_ema + (1 - EMA_BETA) * logits_np

        probs = torch.softmax(torch.tensor(self.logits_ema), dim=0).numpy()
        new_id = int(probs.argmax())
        new_conf = float(probs[new_id])
        s = np.sort(probs)[::-1]
        new_margin = float(s[0] - s[1]) if len(s) > 1 else 1.0

        if self.label_id is None:
            self.label_id = new_id
            self.label_name = LABELS[new_id]
            self.conf = new_conf
            self.streak = 0
            return

        if new_id != self.label_id:
            if (new_conf >= TH_SWITCH) and (new_margin >= MARGIN_TH):
                self.streak += 1
                if self.streak >= SWITCH_K:
                    self.label_id = new_id
                    self.label_name = LABELS[new_id]
                    self.streak = 0
            else:
                self.streak = 0
        else:
            self.streak = 0

        self.conf = new_conf


class LockedTarget:
    def __init__(self, target_id, init_box, init_app=None):
        self.target_id = target_id
        self.filter = BBoxFilter(beta_pos=0.88, beta_size=0.96)
        self.last_box = self.filter.update(init_box)
        self.prev_box = self.last_box.copy()
        self.last_seen = 0
        self.missed = 0
        self.buffers = deque(maxlen=T_BUFFER)
        self.state = TrackState()
        self.app_ref = init_app
        self.app_beta = 0.95

    def update_box(self, raw_box, frame_id, app_feat=None):
        self.prev_box = self.last_box.copy()
        self.last_box = self.filter.update(raw_box)
        self.last_seen = frame_id
        self.missed = 0

        if app_feat is not None:
            if self.app_ref is None:
                self.app_ref = app_feat.copy()
            else:
                self.app_ref = self.app_beta * self.app_ref + (1 - self.app_beta) * app_feat
                n = np.linalg.norm(self.app_ref) + 1e-8
                self.app_ref = self.app_ref / n
        return self.last_box.copy()

    def predict_box(self):
        if self.last_box is None:
            return None
        if self.prev_box is None:
            return self.last_box.copy()
        v = self.last_box - self.prev_box
        v = np.clip(v, -20, 20)
        steps = min(self.missed + 1, 3)
        return self.last_box + steps * v


@torch.no_grad()
def encode_det_appearance(dets, frame_bgr):
    valid = []
    tensors = []

    for i, d in enumerate(dets):
        x1, y1, x2, y2 = map(int, d["box"])
        crop = frame_bgr[y1:y2, x1:x2]
        if crop.size == 0:
            d["app"] = None
            continue
        rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        tensors.append(clip_preprocess(Image.fromarray(rgb)))
        valid.append(i)

    if len(tensors) == 0:
        return dets

    x = torch.stack(tensors).to(device)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
        f = clip_model.encode_image(x)
        f = F.normalize(f, dim=-1)
    feats = f.float().cpu().numpy()

    for i, feat in zip(valid, feats):
        dets[i]["app"] = feat
    return dets


@torch.no_grad()
def infer_track_fast(crops_bgr_list):
    T = len(crops_bgr_list)
    if T < T_BUFFER:
        crops_bgr_list = crops_bgr_list + [crops_bgr_list[-1]] * (T_BUFFER - T)
    crops_bgr_list = crops_bgr_list[-T_BUFFER:]
    T = len(crops_bgr_list)

    idxs_pose = np.linspace(0, T - 1, N_POSE).astype(int).tolist()
    kpts_sparse = []
    valid = 0
    kpts_last = np.zeros((17, 3), dtype=np.float32)

    for t in idxs_pose:
        crop = crops_bgr_list[t]
        kpts, scores = pose_model(crop)
        if kpts is None or len(kpts) == 0:
            kpts_sparse.append(np.zeros((17, 3), dtype=np.float32))
            continue
        kk = np.concatenate([kpts[0], scores[0][:, None]], axis=1).astype(np.float32)
        kpts_sparse.append(kk)
        valid += 1
        kpts_last = kk

    kpts_seq = fill_kpts_to_T(kpts_sparse, idxs_pose, T=T)
    conf = kpts_seq[..., 2]
    mean_conf = float(conf.mean())
    miss_ratio = float((conf < 0.2).mean())
    valid_ratio = float(valid / float(len(idxs_pose)))

    r_pose = torch.tensor([mean_conf, miss_ratio, valid_ratio], dtype=torch.float32)
    kpts_t = torch.from_numpy(kpts_seq).float()
    kpts_t = normalize_kpts_clip(kpts_t)
    ws, te = compute_pose_extra_feats(kpts_t)
    r_pose = torch.cat([r_pose, ws.view(1), te.view(1)], dim=0)

    idxs_img = np.linspace(0, T - 1, N_FRAMES_IMG).astype(int).tolist()
    frames_rgb = [cv2.cvtColor(crops_bgr_list[i], cv2.COLOR_BGR2RGB) for i in idxs_img]
    bl = float(np.mean([blur_score(f) for f in frames_rgb]))
    br = float(np.mean([np.mean(f) for f in frames_rgb]))
    r_img = torch.tensor([math.log1p(bl), br / 255.0], dtype=torch.float32)

    imgs = [clip_preprocess(Image.fromarray(f)) for f in frames_rgb]
    x = torch.stack(imgs).to(device)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
        f = clip_model.encode_image(x)
        f = f / f.norm(dim=-1, keepdim=True)
    img_emb = f.mean(dim=0)
    img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)

    logits, alpha = model(
        kpts_t.unsqueeze(0).to(device),
        img_emb.unsqueeze(0).to(device),
        r_pose.unsqueeze(0).to(device),
        r_img.unsqueeze(0).to(device)
    )
    logits_np = logits[0].detach().cpu().numpy()
    alpha_val = float(alpha[0, 0].item())
    return logits_np, alpha_val, kpts_last


def detect_people(frame, W, H):
    results = yolo.predict(
        frame,
        imgsz=IMG_SIZE,
        conf=CONF_TH,
        iou=IOU_TH,
        classes=[0],
        verbose=False
    )
    boxes = results[0].boxes
    dets = []
    if boxes is None or boxes.xyxy is None:
        return dets

    xyxy = boxes.xyxy.cpu().numpy().astype(int)
    confs = boxes.conf.cpu().numpy() if boxes.conf is not None else np.ones(len(xyxy))

    for (x1, y1, x2, y2), dc in zip(xyxy, confs):
        bw = x2 - x1
        bh = y2 - y1
        px = int(bw * PAD)
        py = int(bh * PAD)
        X1 = max(0, x1 - px)
        Y1 = max(0, y1 - py)
        X2 = min(W - 1, x2 + px)
        Y2 = min(H - 1, y2 + py)
        area = max(1, X2 - X1) * max(1, Y2 - Y1)
        dets.append({
            "box": [X1, Y1, X2, Y2],
            "conf": float(dc),
            "area": area,
            "used": False,
            "app": None
        })
    return dets


def associate_targets_to_dets(targets, dets):
    if len(targets) == 0 or len(dets) == 0:
        return [], list(range(len(targets))), list(range(len(dets)))

    BIG = 1e6
    cost = np.full((len(targets), len(dets)), BIG, dtype=np.float32)

    for ti, tgt in enumerate(targets):
        pred_box = tgt.predict_box()
        if pred_box is None:
            continue

        for di, d in enumerate(dets):
            box = np.array(d["box"], dtype=np.float32)
            iou = bbox_iou(pred_box, box)
            dist = center_dist(pred_box, box)
            spen = size_penalty(pred_box, box)
            conf = d["conf"]

            app_sim = 0.0
            if tgt.app_ref is not None and d["app"] is not None:
                app_sim = cosine_np(tgt.app_ref, d["app"])

            if dist > MAX_CENTER_DIST:
                continue
            if iou < MIN_IOU_MATCH and dist > 35:
                continue
            if spen > 0.9:
                continue
            if tgt.app_ref is not None and d["app"] is not None and app_sim < 0.18:
                continue

            score = (
                2.8 * iou
                + 1.8 * app_sim
                + 0.20 * conf
                - 0.010 * dist
                - 0.45 * spen
            )
            cost[ti, di] = -score

    row_ind, col_ind = linear_sum_assignment(cost)

    matches = []
    unmatched_t = set(range(len(targets)))
    unmatched_d = set(range(len(dets)))

    for r, c in zip(row_ind, col_ind):
        if cost[r, c] >= BIG / 2:
            continue
        matches.append((r, c))
        unmatched_t.discard(r)
        unmatched_d.discard(c)

    return matches, sorted(list(unmatched_t)), sorted(list(unmatched_d))

Cell 6

In [ ]:
# =========================================
# CELL 6 - PROCESS VIDEO JOB + WORKER LOOP
# Chạy 1 lần
# =========================================

def finalize_segment(current_rows, session_id, target_id, seg_counter, video_duration_sec, clips_dir):
    df_seg = pd.DataFrame(current_rows)

    start_frame = int(df_seg["frame"].min())
    end_frame = int(df_seg["frame"].max())
    start_time = float(df_seg["time_sec"].min())
    end_time = float(df_seg["time_sec"].max())

    duration = max(0.0, end_time - start_time)
    if duration < MIN_SEGMENT_SEC:
        return None

    peak_conf = float(df_seg["conf"].max())
    mean_conf = float(df_seg["conf"].mean())

    if CLIP_USE_START_AS_ANCHOR:
        clip_start = start_time
    else:
        center_t = 0.5 * (start_time + end_time)
        clip_start = center_t - (CLIP_DURATION_SEC / 2.0)

    clip_start = clamp(clip_start, 0.0, max(0.0, video_duration_sec - CLIP_DURATION_SEC))
    clip_end = min(video_duration_sec, clip_start + CLIP_DURATION_SEC)
    clip_duration = max(0.0, clip_end - clip_start)

    segment_id = f"seg-{seg_counter:04d}"
    clip_filename = f"{segment_id}_target{target_id}_{PHONE_LABEL}.mp4"

    clip_rel_path = os.path.join("clips", clip_filename)
    clip_abs_path = os.path.join(clips_dir, clip_filename)

    return {
        "session_id": session_id,
        "segment_id": segment_id,
        "target_id": int(target_id),
        "label": PHONE_LABEL,
        "start_frame": start_frame,
        "end_frame": end_frame,
        "start_time_sec": round(start_time, 3),
        "end_time_sec": round(end_time, 3),
        "duration_sec": round(duration, 3),
        "peak_conf": round(peak_conf, 4),
        "mean_conf": round(mean_conf, 4),
        "clip_start_sec": round(clip_start, 3),
        "clip_end_sec": round(clip_end, 3),
        "clip_duration_sec": round(clip_duration, 3),
        "clip_filename": clip_filename,
        "clip_path": clip_rel_path,
        "clip_abs_path": clip_abs_path,
        "telegram_ready": 0,
    }


def build_phone_segments(df, session_id, video_duration_sec, clips_dir):
    columns = [
        "session_id", "segment_id", "target_id", "label",
        "start_frame", "end_frame",
        "start_time_sec", "end_time_sec", "duration_sec",
        "peak_conf", "mean_conf",
        "clip_start_sec", "clip_end_sec", "clip_duration_sec",
        "clip_filename", "clip_path", "clip_abs_path",
        "telegram_ready"
    ]

    if len(df) == 0:
        return pd.DataFrame(columns=columns)

    df = df.copy()
    df = df[df["event_candidate"] == 1].copy()
    if len(df) == 0:
        return pd.DataFrame(columns=columns)

    df = df.sort_values(["target_id", "time_sec"]).reset_index(drop=True)

    rows = []
    seg_counter = 0

    for target_id, g in df.groupby("target_id"):
        g = g.sort_values("time_sec").reset_index(drop=True)
        current = []
        prev_t = None

        for _, row in g.iterrows():
            t = float(row["time_sec"])
            if prev_t is None:
                current = [row]
            else:
                gap = t - prev_t
                if gap <= SEGMENT_GAP_SEC:
                    current.append(row)
                else:
                    if len(current) > 0:
                        seg_counter += 1
                        seg_row = finalize_segment(
                            current,
                            session_id,
                            int(target_id),
                            seg_counter,
                            video_duration_sec,
                            clips_dir,
                        )
                        if seg_row is not None:
                            rows.append(seg_row)
                    current = [row]
            prev_t = t

        if len(current) > 0:
            seg_counter += 1
            seg_row = finalize_segment(
                current,
                session_id,
                int(target_id),
                seg_counter,
                video_duration_sec,
                clips_dir,
            )
            if seg_row is not None:
                rows.append(seg_row)

    if len(rows) == 0:
        return pd.DataFrame(columns=columns)

    return pd.DataFrame(rows, columns=columns)


def process_video_job(job_dir):
    job_json_path = os.path.join(job_dir, "job.json")
    if not os.path.exists(job_json_path):
        raise FileNotFoundError(f"Missing job.json: {job_json_path}")

    job = read_json(job_json_path)

    job_id = job["job_id"]
    session_id = job.get("session_id", job_id)
    input_video_name = job.get("input_video", "input.mp4")
    input_video_path = os.path.join(job_dir, input_video_name)

    if not os.path.exists(input_video_path):
        raise FileNotFoundError(f"Input video not found: {input_video_path}")

    write_status(job_dir, job_id, session_id, "processing", 5, "Reading video metadata")

    out_video_raw = os.path.join(job_dir, "annotated_video_raw.mp4")
    out_video = os.path.join(job_dir, "annotated_video.mp4")
    out_csv = os.path.join(job_dir, "events_frame.csv")
    out_seg_csv = os.path.join(job_dir, "phone_segments.csv")
    out_json = os.path.join(job_dir, "result.json")
    clips_dir = os.path.join(job_dir, "clips")
    os.makedirs(clips_dir, exist_ok=True)

    cap_meta = cv2.VideoCapture(input_video_path)
    if not cap_meta.isOpened():
        raise RuntimeError(f"Cannot open input video: {input_video_path}")

    fps_in = cap_meta.get(cv2.CAP_PROP_FPS)
    W = int(cap_meta.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap_meta.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap_meta.get(cv2.CAP_PROP_FRAME_COUNT))
    video_duration_sec = total_frames / fps_in if fps_in and fps_in > 0 else 0.0
    cap_meta.release()

    if fps_in is None or fps_in <= 0:
        raise RuntimeError("Invalid FPS from input video")

    max_frames = int(MAX_SECONDS * fps_in) if MAX_SECONDS is not None else None

    # ===== INIT TARGETS =====
    write_status(job_dir, job_id, session_id, "processing", 10, "Initializing targets")

    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video for init: {input_video_path}")

    ok, first_frame = cap.read()
    if not ok or first_frame is None:
        cap.release()
        raise RuntimeError("Cannot read first frame")

    init_dets = detect_people(first_frame, W, H)
    init_dets = sorted(init_dets, key=lambda d: d["area"], reverse=True)

    filtered_dets = []
    for d in init_dets:
        overlap = False
        for fd in filtered_dets:
            if bbox_iou(d["box"], fd["box"]) > 0.3:
                overlap = True
                break
        if not overlap:
            filtered_dets.append(d)
        if len(filtered_dets) == NUM_TARGETS:
            break

    filtered_dets = encode_det_appearance(filtered_dets, first_frame)

    targets = []
    for i, d in enumerate(filtered_dets):
        t = LockedTarget(i, d["box"], init_app=d["app"])
        x1, y1, x2, y2 = map(int, t.last_box)

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(W - 1, x2)
        y2 = min(H - 1, y2)

        crop = first_frame[y1:y2, x1:x2].copy()
        if crop.size > 0:
            t.buffers.append(crop)
            logits_np, alpha_val, kpts_last = infer_track_fast([crop])
            t.state.update_logits(logits_np, alpha_val, kpts_last)
        targets.append(t)

    cap.release()

    # ===== MAIN LOOP =====
    write_status(job_dir, job_id, session_id, "processing", 20, "Running inference")

    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot reopen video: {input_video_path}")

    out_fps = fps_in / FRAME_STRIDE if FRAME_STRIDE > 0 else fps_in
    if out_fps <= 0:
        out_fps = fps_in

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out_video_raw, fourcc, out_fps, (W, H))

    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Cannot create output video: {out_video_raw}")

    logs = []
    frame_idx = 0
    kept_idx = 0

    t_last = time.time()
    fps_smooth = None
    last_progress = -1

    print(f"Running job={job_id}, session={session_id}, targets={len(targets)}")

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if max_frames is not None and frame_idx >= max_frames:
                break

            if frame_idx % FRAME_STRIDE != 0:
                frame_idx += 1
                continue

            dets = detect_people(frame, W, H)
            dets = encode_det_appearance(dets, frame)

            matches, unmatched_t, unmatched_d = associate_targets_to_dets(targets, dets)

            for ti, di in matches:
                tgt = targets[ti]
                d = dets[di]
                box_smooth = tgt.update_box(d["box"], kept_idx, app_feat=d["app"])

                x1, y1, x2, y2 = map(int, box_smooth)
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(W - 1, x2)
                y2 = min(H - 1, y2)

                crop = frame[y1:y2, x1:x2].copy()
                if crop.size > 0:
                    tgt.buffers.append(crop)

            for ti in unmatched_t:
                targets[ti].missed += 1

            if kept_idx % PRED_STRIDE == 0:
                for tgt in targets:
                    if len(tgt.buffers) >= 1:
                        logits_np, alpha_val, kpts_last = infer_track_fast(list(tgt.buffers))
                        tgt.state.update_logits(logits_np, alpha_val, kpts_last)

                        box_now = tgt.last_box if tgt.last_box is not None else np.array([0, 0, 0, 0], dtype=np.float32)
                        x1, y1, x2, y2 = map(float, box_now.tolist())

                        label_name = str(tgt.state.label_name)
                        conf_val = float(tgt.state.conf)
                        is_phone = int(label_name == PHONE_LABEL)
                        event_candidate = int(label_name == PHONE_LABEL and conf_val >= PHONE_CONF_TH)

                        logs.append({
                            "session_id": session_id,
                            "job_id": job_id,
                            "frame": int(frame_idx),
                            "time_sec": float(frame_idx / fps_in),
                            "target_id": int(tgt.target_id),
                            "x1": x1,
                            "y1": y1,
                            "x2": x2,
                            "y2": y2,
                            "label": label_name,
                            "conf": conf_val,
                            "alpha": float(tgt.state.alpha),
                            "missed": int(tgt.missed),
                            "track_visible": int(tgt.missed == 0),
                            "is_phone": is_phone,
                            "event_candidate": event_candidate,
                            "source_fps": float(fps_in),
                            "frame_stride": int(FRAME_STRIDE),
                            "pred_stride": int(PRED_STRIDE),
                        })

            for tgt in targets:
                box = None
                ghost = False

                if tgt.missed == 0 and tgt.last_box is not None:
                    box = tgt.last_box.copy()
                elif 1 <= tgt.missed <= GHOST_MAX_GAP:
                    pb = tgt.predict_box()
                    if pb is not None:
                        box = pb
                        ghost = True

                if box is None:
                    continue

                x1, y1, x2, y2 = map(int, box.tolist())
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(W - 1, x2)
                y2 = min(H - 1, y2)

                if ghost:
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (120, 120, 120), 2)
                    txt = f"T{tgt.target_id} ghost({tgt.missed})"
                    draw_tag(frame, x1, y1, txt, (120, 120, 120))
                else:
                    col = label_color(tgt.state.label_name)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), col, 2)
                    txt = f"T{tgt.target_id} {tgt.state.label_name} p={tgt.state.conf:.2f}"
                    draw_tag(frame, x1, y1, txt, col)

            now_t = time.time()
            dt = now_t - t_last
            t_last = now_t
            inst_fps = (1.0 / dt) if dt > 1e-6 else 0.0
            fps_smooth = inst_fps if fps_smooth is None else (FPS_EMA * fps_smooth + (1 - FPS_EMA) * inst_fps)

            if SHOW_FPS_ON_VIDEO:
                cv2.putText(frame, f"FPS {fps_smooth:.1f}", (20, 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 4, cv2.LINE_AA)
                cv2.putText(frame, f"FPS {fps_smooth:.1f}", (20, 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2, cv2.LINE_AA)

            writer.write(frame)

            if total_frames > 0:
                progress = 20 + int(50 * min(1.0, frame_idx / total_frames))
                if progress != last_progress and progress % 5 == 0:
                    write_status(
                        job_dir,
                        job_id,
                        session_id,
                        "processing",
                        progress,
                        f"Processing frame {frame_idx}/{total_frames}"
                    )
                    last_progress = progress

            kept_idx += 1
            frame_idx += 1

    finally:
        cap.release()
        writer.release()

    # ===== TRANSCODE WEB VIDEO =====
    write_status(job_dir, job_id, session_id, "processing", 74, "Converting annotated video for web")

    ok_web, err_web = ffmpeg_make_web_video(out_video_raw, out_video)
    if not ok_web:
        print(f"[WARN] Failed to transcode annotated video: {err_web}")
        if os.path.exists(out_video_raw) and not os.path.exists(out_video):
            shutil.copy2(out_video_raw, out_video)

    try:
        if os.path.exists(out_video_raw):
            os.remove(out_video_raw)
    except Exception:
        pass

    # ===== SAVE CSV =====
    write_status(job_dir, job_id, session_id, "processing", 78, "Saving frame events")
    df_events = pd.DataFrame(logs)
    df_events.to_csv(out_csv, index=False, encoding="utf-8-sig")

    # ===== BUILD SEGMENTS =====
    write_status(job_dir, job_id, session_id, "processing", 84, "Building phone segments")
    df_segments = build_phone_segments(df_events, session_id, video_duration_sec, clips_dir)
    df_segments.to_csv(out_seg_csv, index=False, encoding="utf-8-sig")

    # ===== CUT CLIPS =====
    write_status(job_dir, job_id, session_id, "processing", 90, "Cutting 5-second clips")

    if len(df_segments) > 0:
        for i, row in df_segments.iterrows():
            clip_abs_path = row["clip_abs_path"]
            ok_cut, err = ffmpeg_cut_clip(
                input_video=input_video_path,
                out_path=clip_abs_path,
                start_sec=row["clip_start_sec"],
                duration_sec=row["clip_duration_sec"]
            )
            if ok_cut:
                df_segments.loc[i, "telegram_ready"] = 1
            else:
                print(f"[WARN] Failed to cut clip for {row['segment_id']}: {err}")

    df_segments.to_csv(out_seg_csv, index=False, encoding="utf-8-sig")

    # ===== BUILD RESULT JSON =====
    write_status(job_dir, job_id, session_id, "processing", 96, "Building result JSON")

    summary = {
        "session_id": session_id,
        "job_id": job_id,
        "video_name": os.path.basename(input_video_path),
        "video_duration_sec": round(video_duration_sec, 3),
        "num_targets_locked": int(len(targets)),
        "num_frame_events": int(len(df_events)),
        "num_phone_rows": int(df_events["is_phone"].sum()) if len(df_events) > 0 and "is_phone" in df_events.columns else 0,
        "num_phone_segments": int(len(df_segments)),
        "num_phone_clips_ready": int(df_segments["telegram_ready"].sum()) if len(df_segments) > 0 and "telegram_ready" in df_segments.columns else 0,
    }

    label_distribution = (
        df_events["label"].value_counts().to_dict()
        if len(df_events) > 0 and "label" in df_events.columns
        else {}
    )

    events_json = []
    if len(df_segments) > 0:
        for _, row in df_segments.iterrows():
            events_json.append({
                "id": row["segment_id"],
                "target_id": int(row["target_id"]),
                "label": row["label"],
                "confidence": float(row["peak_conf"]),
                "start_time_sec": float(row["start_time_sec"]),
                "end_time_sec": float(row["end_time_sec"]),
                "clip_start_sec": float(row["clip_start_sec"]),
                "clip_end_sec": float(row["clip_end_sec"]),
                "clip_path": row["clip_path"],
                "telegram_ready": bool(row["telegram_ready"]),
            })

    result = {
        "job_id": job_id,
        "session_id": session_id,
        "video_name": os.path.basename(input_video_path),
        "annotated_video_path": "annotated_video.mp4",
        "events_frame_csv_path": "events_frame.csv",
        "phone_segments_csv_path": "phone_segments.csv",
        "clips_dir": "clips",
        "summary": summary,
        "label_distribution": label_distribution,
        "events": events_json
    }

    write_json(out_json, result)

    write_status(
        job_dir,
        job_id,
        session_id,
        "done",
        100,
        "Inference completed successfully",
        {
            "result_json_path": "result.json",
            "annotated_video_path": "annotated_video.mp4"
        }
    )


def is_job_dir_ready(job_dir):
    if not os.path.isdir(job_dir):
        return False

    job_json_path = os.path.join(job_dir, "job.json")
    input_video_path = os.path.join(job_dir, "input.mp4")

    if not os.path.exists(job_json_path):
        return False
    if not os.path.exists(input_video_path):
        return False

    return True


def list_incoming_jobs():
    if not os.path.exists(INCOMING_DIR):
        return []

    jobs = []
    for name in sorted(os.listdir(INCOMING_DIR)):
        job_dir = os.path.join(INCOMING_DIR, name)
        if not os.path.isdir(job_dir):
            continue
        if not is_job_dir_ready(job_dir):
            continue
        jobs.append(job_dir)

    return jobs


def claim_job(incoming_job_dir):
    job_name = os.path.basename(incoming_job_dir)
    processing_job_dir = os.path.join(PROCESSING_DIR, job_name)

    if os.path.exists(processing_job_dir):
        try:
            shutil.rmtree(processing_job_dir)
        except Exception:
            return None

    shutil.move(incoming_job_dir, processing_job_dir)

    if not is_job_dir_ready(processing_job_dir):
        print(f"[WARN] Claimed job but missing required files: {processing_job_dir}")
        try:
            failed_job_dir = os.path.join(FAILED_DIR, job_name)
            if os.path.exists(failed_job_dir):
                shutil.rmtree(failed_job_dir)
            shutil.move(processing_job_dir, failed_job_dir)
            print(f"[WARN] Moved invalid job to failed: {failed_job_dir}")
        except Exception as e:
            print(f"[WARN] Failed to move invalid job to failed: {e}")
        return None

    return processing_job_dir


def finalize_job_success(processing_job_dir):
    job_name = os.path.basename(processing_job_dir)
    done_job_dir = os.path.join(DONE_DIR, job_name)

    if os.path.exists(done_job_dir):
        shutil.rmtree(done_job_dir)

    shutil.move(processing_job_dir, done_job_dir)
    return done_job_dir


def finalize_job_failed(processing_job_dir):
    job_name = os.path.basename(processing_job_dir)
    failed_job_dir = os.path.join(FAILED_DIR, job_name)

    if os.path.exists(failed_job_dir):
        shutil.rmtree(failed_job_dir)

    shutil.move(processing_job_dir, failed_job_dir)
    return failed_job_dir


def worker_loop():
    print("Colab worker started.")
    print("Watching:", INCOMING_DIR)

    while KEEP_RUNNING:
        try:
            incoming_jobs = list_incoming_jobs()

            if not incoming_jobs:
                print(f"[{now_iso()}] No jobs. Sleeping {POLL_INTERVAL_SEC}s...")
                time.sleep(POLL_INTERVAL_SEC)
                continue

            for incoming_job_dir in incoming_jobs:
                processing_job_dir = claim_job(incoming_job_dir)
                if processing_job_dir is None:
                    continue

                job_json_path = os.path.join(processing_job_dir, "job.json")
                if not os.path.exists(job_json_path):
                    print(f"[{now_iso()}] Missing job.json after claim: {processing_job_dir}")
                    try:
                        finalize_job_failed(processing_job_dir)
                    except Exception as e:
                        print(f"[WARN] Failed to move missing-job.json case to failed: {e}")
                    continue

                try:
                    job = read_json(job_json_path)
                except Exception as e:
                    print(f"[{now_iso()}] Failed to read job.json: {job_json_path}")
                    print(traceback.format_exc())

                    try:
                        failed_job_dir = finalize_job_failed(processing_job_dir)
                        print(f"Moved unreadable job to: {failed_job_dir}")
                    except Exception as move_err:
                        print(f"[WARN] Failed moving unreadable job: {move_err}")
                    continue

                job_id = job.get("job_id", os.path.basename(processing_job_dir))
                session_id = job.get("session_id", job_id)

                try:
                    write_status(
                        processing_job_dir,
                        job_id,
                        session_id,
                        "processing",
                        1,
                        "Job claimed by Colab worker"
                    )
                    print(f"[{now_iso()}] Processing job_id={job_id}, session_id={session_id}")

                    process_video_job(processing_job_dir)

                    done_job_dir = finalize_job_success(processing_job_dir)
                    print(f"[{now_iso()}] Done job_id={job_id} -> {done_job_dir}")

                except Exception as e:
                    err_text = traceback.format_exc()
                    print(f"[{now_iso()}] FAILED job_id={job_id}")
                    print(err_text)

                    try:
                        write_status(
                            processing_job_dir,
                            job_id,
                            session_id,
                            "failed",
                            100,
                            str(e),
                            {"error": err_text[:4000]}
                        )
                    except Exception as status_err:
                        print(f"[WARN] Failed to write failed status: {status_err}")

                    try:
                        failed_job_dir = finalize_job_failed(processing_job_dir)
                        print(f"Moved failed job to: {failed_job_dir}")
                    except Exception as move_err:
                        print(f"[WARN] Failed to move failed job: {move_err}")

        except Exception:
            print(traceback.format_exc())
            time.sleep(POLL_INTERVAL_SEC)

Cell 7

In [ ]:
# =========================================
# CELL 7 - START WORKER
# Cell này phải chạy liên tục
# =========================================
worker_loop()

Colab worker started.
Watching: /content/drive/MyDrive/classroom_jobs/incoming
[2026-03-14T06:13:09Z] No jobs. Sleeping 5s...


/tmp/ipykernel_666/2943754828.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat(timespec="seconds") + "Z"


[2026-03-14T06:13:14Z] No jobs. Sleeping 5s...
[2026-03-14T06:13:19Z] No jobs. Sleeping 5s...
[2026-03-14T06:13:24Z] No jobs. Sleeping 5s...
[2026-03-14T06:13:29Z] No jobs. Sleeping 5s...
[2026-03-14T06:13:34Z] No jobs. Sleeping 5s...
[2026-03-14T06:13:39Z] No jobs. Sleeping 5s...
[2026-03-14T06:13:45Z] Processing job_id=classroom_a_20260314_131326, session_id=classroom_a_20260314_131326
Running job=classroom_a_20260314_131326, session=classroom_a_20260314_131326, targets=5


/tmp/ipykernel_666/2943754828.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat(timespec="seconds") + "Z"
/tmp/ipykernel_666/2943754828.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat(timespec="seconds") + "Z"
/tmp/ipykernel_666/2943754828.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat(timespec="seconds") + "Z"


[2026-03-14T06:14:56Z] Done job_id=classroom_a_20260314_131326 -> /content/drive/MyDrive/classroom_jobs/done/classroom_a_20260314_131326
[2026-03-14T06:14:56Z] No jobs. Sleeping 5s...
[2026-03-14T06:15:01Z] No jobs. Sleeping 5s...
[2026-03-14T06:15:06Z] No jobs. Sleeping 5s...
[2026-03-14T06:15:11Z] No jobs. Sleeping 5s...
[2026-03-14T06:15:16Z] No jobs. Sleeping 5s...
[2026-03-14T06:15:21Z] No jobs. Sleeping 5s...
[2026-03-14T06:15:26Z] No jobs. Sleeping 5s...
[2026-03-14T06:15:31Z] No jobs. Sleeping 5s...
[2026-03-14T06:15:36Z] No jobs. Sleeping 5s...
[2026-03-14T06:15:41Z] No jobs. Sleeping 5s...
[2026-03-14T06:15:46Z] No jobs. Sleeping 5s...
[2026-03-14T06:15:51Z] No jobs. Sleeping 5s...
[2026-03-14T06:15:56Z] No jobs. Sleeping 5s...
[2026-03-14T06:16:01Z] No jobs. Sleeping 5s...
[2026-03-14T06:16:06Z] No jobs. Sleeping 5s...
[2026-03-14T06:16:11Z] No jobs. Sleeping 5s...
[2026-03-14T06:16:16Z] No jobs. Sleeping 5s...
[2026-03-14T06:16:21Z] No jobs. Sleeping 5s...
[2026-03-14T06:16

/tmp/ipykernel_666/2943754828.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat(timespec="seconds") + "Z"
/tmp/ipykernel_666/2943754828.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat(timespec="seconds") + "Z"
/tmp/ipykernel_666/2943754828.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat(timespec="seconds") + "Z"


[2026-03-14T06:23:51Z] Processing job_id=classroom_b_20260314_132251, session_id=classroom_b_20260314_132251
Running job=classroom_b_20260314_132251, session=classroom_b_20260314_132251, targets=5


/tmp/ipykernel_666/2943754828.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat(timespec="seconds") + "Z"
/tmp/ipykernel_666/2943754828.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat(timespec="seconds") + "Z"
/tmp/ipykernel_666/2943754828.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat(timespec="seconds") + "Z"


[2026-03-14T06:25:00Z] Done job_id=classroom_b_20260314_132251 -> /content/drive/MyDrive/classroom_jobs/done/classroom_b_20260314_132251
[2026-03-14T06:25:00Z] No jobs. Sleeping 5s...
[2026-03-14T06:25:05Z] No jobs. Sleeping 5s...
[2026-03-14T06:25:10Z] No jobs. Sleeping 5s...
[2026-03-14T06:25:15Z] No jobs. Sleeping 5s...
[2026-03-14T06:25:20Z] No jobs. Sleeping 5s...
[2026-03-14T06:25:25Z] No jobs. Sleeping 5s...
[2026-03-14T06:25:30Z] No jobs. Sleeping 5s...
[2026-03-14T06:25:35Z] No jobs. Sleeping 5s...
[2026-03-14T06:25:40Z] No jobs. Sleeping 5s...
[2026-03-14T06:25:45Z] No jobs. Sleeping 5s...
[2026-03-14T06:25:50Z] No jobs. Sleeping 5s...
[2026-03-14T06:25:55Z] No jobs. Sleeping 5s...
[2026-03-14T06:26:00Z] No jobs. Sleeping 5s...
[2026-03-14T06:26:05Z] No jobs. Sleeping 5s...
[2026-03-14T06:26:10Z] No jobs. Sleeping 5s...
[2026-03-14T06:26:15Z] No jobs. Sleeping 5s...
[2026-03-14T06:26:20Z] No jobs. Sleeping 5s...
[2026-03-14T06:26:25Z] No jobs. Sleeping 5s...
[2026-03-14T06:26

In [ ]:
import os

base = "/content/drive/MyDrive/classroom_jobs/incoming"
print("Exists:", os.path.exists(base))
print("Items:", os.listdir(base) if os.path.exists(base) else "missing")